In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore") # Suppress warnings, especially from LogisticRegression convergence

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('marriage.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'marriage.csv' not found. Please ensure it's in the same directory.")
    # Create dummy data for demonstration if file is not found
    print("Generating dummy data for demonstration purposes.")
    num_participants = 170
    num_features = 54
    dummy_data = np.random.rand(num_participants, num_features - 1) * 5 # Simulate feature values
    dummy_labels = np.random.randint(0, 2, num_participants).reshape(-1, 1) # Simulate labels 0 or 1
    df = pd.DataFrame(np.hstack((dummy_data, dummy_labels)), columns=[f'feature_{i}' for i in range(num_features - 1)] + ['label'])
    print("Dummy data generated.")

# Separate features (X) and target (y)
X = df.iloc[:, :-1]  # All columns except the last one
y = df.iloc[:, -1]   # The last column (label)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFirst 5 rows of features (X):")
print(X.head())
print("\nFirst 5 values of target (y):")
print(y.head())

In [ ]:
# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Using stratify=y ensures that the proportion of classes in the train and test sets is similar to the original dataset.

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nDistribution of classes in y_train:")
print(y_train.value_counts(normalize=True))
print("\nDistribution of classes in y_test:")
print(y_test.value_counts(normalize=True))

In [ ]:
# Gaussian Naive Bayes Classifier
print("--- Gaussian Naive Bayes ---")

# Custom variance handling for Naive Bayes (as per remark)
# We will estimate variances from training data and then adjust
epsilon = 1e-3

# Calculate variance for each feature in X_train
variances = X_train.var()

# Identify features with variance close to zero and adjust
# Note: GaussianNB has a `var_smoothing` parameter, but the problem implies manual handling.
# If `var_smoothing` is used, it adds this value to the variance for numerical stability globally.
# Here, we are explicitly checking and adjusting zero/near-zero variances before passing to the model.
# However, scikit-learn's GaussianNB handles this internally with `var_smoothing` by default (usually a very small value).
# If we were to implement Naive Bayes from scratch, we'd apply this.
# For scikit-learn, the simplest way to strictly follow "set variance to small number" for each feature
# is to manually calculate parameters or preprocess X_train.
# Let's check how GaussianNB internally estimates variance. It typically adds a `var_smoothing`
# term. The prompt's wording "if the variance is zero... set the variance to be a small number"
# suggests a more direct intervention on the estimated variances.

# A common way to implement the spirit of the remark for scikit-learn's GaussianNB
# is to apply var_smoothing, or directly modify the X_train by adding noise if variance is too low.
# However, modifying X_train can be complex. The simplest direct application to GaussianNB
# is via `var_smoothing`. Let's use it as it's designed for this purpose.
# If explicit manual variance calculation and replacement is required, that would mean
# reimplementing GaussianNB likelihood computations. For now, var_smoothing is the standard.
# If the intention was to specifically estimate variances, modify them, and then pass them
# to a custom Naive Bayes implementation, that's a different task.
# Given the use of `scikit-learn` for other classifiers, `var_smoothing` is the most
# direct interpretation of the requirement for `GaussianNB`.

gnb = GaussianNB(var_smoothing=epsilon) # Adds epsilon to the variance of each feature
gnb.fit(X_train, y_train)
y_pred_gnb = gnb.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_gnb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_gnb))
print("\nConfusion Matrix:")
cm_gnb = confusion_matrix(y_test, y_pred_gnb)
print(cm_gnb)

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_gnb, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted No Divorce', 'Predicted Divorce'], yticklabels=['Actual No Divorce', 'Actual Divorce'])
plt.title('Confusion Matrix - Gaussian Naive Bayes')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# Logistic Regression Classifier
print("\n--- Logistic Regression ---")

# Using default solver 'lbfgs' or 'liblinear' often works well.
# 'liblinear' is good for small datasets and L1/L2 regularization.
# 'lbfgs' is a good default. Max_iter increased for convergence.
log_reg = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_log_reg):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log_reg))
print("\nConfusion Matrix:")
cm_log_reg = confusion_matrix(y_test, y_pred_log_reg)
print(cm_log_reg)

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_log_reg, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted No Divorce', 'Predicted Divorce'], yticklabels=['Actual No Divorce', 'Actual Divorce'])
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# K-Nearest Neighbors Classifier
print("\n--- K-Nearest Neighbors (KNN) ---")

# Choosing n_neighbors: A common practice is sqrt(N) or odd numbers.
# For N=170, sqrt(170) is approx 13. Let's try 5 as a starting point.
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))
print("\nConfusion Matrix:")
cm_knn = confusion_matrix(y_test, y_pred_knn)
print(cm_knn)

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted No Divorce', 'Predicted Divorce'], yticklabels=['Actual No Divorce', 'Actual Divorce'])
plt.title('Confusion Matrix - K-Nearest Neighbors (KNN)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
print("\n--- Summary of Classifier Performance ---")
print(f"Gaussian Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_gnb):.4f}")
print(f"Logistic Regression Accuracy:  {accuracy_score(y_test, y_pred_log_reg):.4f}")
print(f"K-Nearest Neighbors (KNN) Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")

# You can also compare F1-scores, precision, recall etc.
# For simplicity, let's just stick to accuracy for a quick comparison.